# Resistivity inversion with synthetic models

**Near-Surface Geophysics — Missouri S&T**

With field data you never know what the ground really looks like, so you
cannot tell how much of an inverted image is the ground and how much is the
inversion. A **synthetic test** fixes that: you make up the ground, compute
the data a survey over it would record, add noise, invert those data, and
compare the image with the ground you started from.

This notebook does that for two kinds of ground that matter in Missouri:

- **A cave or mine opening**: an air-filled void in limestone under layered
  soil and clay. Air does not conduct, so the void is very **resistive**.
- **Soil over karsted limestone**: clay-rich soil filling an irregular,
  pinnacled rock surface, with a soil-filled sinkhole throat and a small void
  in the rock.

The sections:

1. **The survey** and the grid the inversion solves for.
2. **The true models**, as Python functions you can edit.
3. **The forward problem**: synthetic apparent resistivities, plus noise.
4. **The inversion** and its smoothness parameter λ, run for a range of λ.
5. **Explore with a slider**: true against inverted resistivity, observed
   against predicted data, as λ changes.
6. **Choosing λ**: the misfit, the L-curve, and what the truth says.
7. **What a smooth inversion cannot recover.**
8. **Your turn**: a live inversion and things to try.

Units are **metres** and **ohm-metres** (Ω·m) throughout. Everything uses the
`shallowgeo.ert` module (NumPy and SciPy only), the same code as the
`ert_supersting.ipynb` field-data notebook.

> **The one thing to take away.** Many different images fit the same data.
> The smoothness setting picks one of them, and the noise level of the data
> is what tells you how smooth that image should be.

Questions marked **Q** are for your write-up.

## 0. Setup

Run this cell first. It installs `shallowgeo` if it is missing (or out of
date) and imports what the notebook uses.

In [ ]:
# Checks for the *ERT* code specifically: an older install imports fine and
# then lacks it, and since the version string does not change between builds,
# a plain `pip install` reports "already satisfied".
import importlib
import sys

PACKAGE = ("shallow-geophysics @ "
           "git+https://github.com/Maurer-GEMLab/shallow-geophysics.git")

def shallowgeo_ready():
    for name in [n for n in sys.modules
                 if n == "shallowgeo" or n.startswith("shallowgeo.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    try:
        from shallowgeo.ert import lambda_sweep, resistivity_tomography  # noqa: F401
        return True
    except ImportError:
        return False

if not shallowgeo_ready():
    %pip install -q --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    %pip install -q --force-reinstall --no-deps --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    raise ImportError(
        "shallowgeo.ert is still missing after installing. "
        "In Colab: Runtime > Restart session, then run this cell again."
    )
try:
    import ipywidgets  # noqa: F401  (ships with Colab)
except ImportError:
    %pip install -q ipywidgets

import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import FormatStrFormatter, LogLocator, NullFormatter
from IPython.display import display
from ipywidgets import (interact, interact_manual, Checkbox, Dropdown, FloatLogSlider,
                        FloatSlider, SelectionSlider)

from shallowgeo.ert import (
    dipole_dipole, wenner, schlumberger, make_mesh, apparent_resistivity,
    resistivity_tomography, lambda_sweep,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": False})
print("ready")

## 1. The survey

A line of **48 electrodes, 2 m apart** (94 m long), recorded with a
**dipole-dipole** scheme: current goes in through two neighbouring electrodes
(A, B) and the voltage is measured between two others (M, N) further along
the line. The dipole length *a* is 1, 2 or 3 electrode spacings and the
separation *n* runs from 1 to 6 dipole lengths, which gives 666 readings.
Each reading becomes an **apparent resistivity**

$$\rho_a = k \, \frac{\Delta V}{I},$$

the resistivity a uniform ground would need to give the same voltage, where
the geometric factor *k* depends only on where the four electrodes are.

These settings are used everywhere below; change them and `Runtime > Run all`.

| setting | what it does |
|---|---|
| `N_ELECTRODES`, `SPACING` | number of electrodes and the distance between them (m) |
| `ARRAY` | `"dipole-dipole"`, `"schlumberger"` (Wenner-Schlumberger) or `"wenner"` |
| `NOISE` | relative noise added to every reading: 0.03 is 3 % |
| `SEED` | the random-number seed; change it for a different draw of the same noise |

In [ ]:
N_ELECTRODES = 48
SPACING = 2.0          # m
ARRAY = "dipole-dipole"
NOISE = 0.03           # 3 % Gaussian noise
SEED = 1


def make_survey(array, n_electrodes, spacing):
    if array == "dipole-dipole":
        return dipole_dipole(n_electrodes, spacing, n_max=6, a_max=3)
    if array == "schlumberger":
        return schlumberger(n_electrodes, spacing, n_max=6, a_max=3)
    if array == "wenner":
        return wenner(n_electrodes, spacing)
    raise ValueError(f"unknown array {array!r}")


survey = make_survey(ARRAY, N_ELECTRODES, SPACING)
print(survey.summary()[["array", "measurements", "electrodes", "spacing_m",
                       "line_length_m", "max_pseudo_depth_m"]])

Each reading is plotted at the midpoint of its four electrodes and at its
**median depth of investigation** (half of the ground's response to that
reading comes from above this depth, half from below). This *pseudosection*
shows where the data are, not what the ground is.

The inversion solves for one resistivity in each cell of a **model grid**,
drawn below it: cells half an electrode spacing wide, thickening with depth,
down to about 1.5 times the deepest median depth.

In [ ]:
inv_mesh = make_mesh(survey)
print(inv_mesh)

d = survey.data
ab = SPACING * (d["b"] - d["a"]).abs()
fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
sc = axes[0].scatter(d["x"], d["depth"], c=ab, s=14, cmap="viridis")
fig.colorbar(sc, ax=axes[0], label="A–B separation (m)", shrink=0.8, pad=0.02)
axes[0].plot(survey.electrodes[:, 0], 0 * survey.electrodes[:, 0], "kv", ms=4, clip_on=False)
axes[0].set_ylim(d["depth"].max() * 1.1, 0)
axes[0].set_ylabel("pseudo-depth (m)")
axes[0].set_title(f"{len(survey)} {ARRAY} readings", fontsize=10)

for z in inv_mesh.z_edges:
    axes[1].plot(inv_mesh.x_edges[[0, -1]], [z, z], "k-", lw=0.3)
for x in inv_mesh.x_edges:
    axes[1].plot([x, x], inv_mesh.z_edges[[0, -1]], "k-", lw=0.3)
axes[1].plot(survey.electrodes[:, 0], 0 * survey.electrodes[:, 0], "kv", ms=4, clip_on=False)
axes[1].set_ylim(inv_mesh.z_edges[-1], 0)
axes[1].set_aspect("equal")
axes[1].set_xlabel("position along line (m)"); axes[1].set_ylabel("depth (m)")
axes[1].set_title(f"model grid: {inv_mesh.shape[0]} × {inv_mesh.shape[1]} = "
                  f"{inv_mesh.n_cells} unknowns", fontsize=10)
plt.tight_layout(); plt.show()

**Q1.** How many unknowns does the inversion have, and how many readings?
What does that tell you about whether the data alone can decide every cell?

## 2. The true models

Each model is a function `rho(X, Z)` that returns the resistivity at
positions `X` along the line and depths `Z`. The numbers are in the two
dictionaries `CAVE` and `KARST`: edit them and re-run from here.

**Cave (or mine opening) in layered ground**

| unit | where | resistivity (Ω·m) |
|---|---|---|
| topsoil | 0–1 m | 100 |
| wet clay (residuum) | 1–3 m | 50 |
| limestone | below 3 m | 500 |
| air-filled cave | 10 m wide, 4 m high, roof 4 m deep, centred at 47 m | 10 000 |

**Soil over karsted limestone**

| unit | where | resistivity (Ω·m) |
|---|---|---|
| dry topsoil | 0–0.8 m | 80 |
| clay-rich residual soil | down to the top of rock: 2–6 m, and 13 m in a sinkhole throat at 35 m | 25 |
| epikarst (weathered, fractured top of rock) | the 2 m below the top of rock | 300 |
| competent limestone | below that | 1500 |
| small air-filled void | 5 m wide, 3 m high, centred 9 m deep at 68 m | 10 000 |

These are typical values, not rules: clay is 5–50 Ω·m, limestone a few
hundred to several thousand depending on its fractures and water, and an
air-filled void is effectively infinite (10 000 Ω·m is already far beyond
anything the data can tell apart, as section 7 shows).

In [ ]:
CAVE = dict(soil=100.0, clay=50.0, rock=500.0, void=1e4,
            soil_base=1.0, clay_base=3.0,                   # depths of layer bottoms (m)
            cave_x=47.0, cave_depth=6.0,                    # centre of the cave (m)
            cave_width=10.0, cave_height=4.0)

KARST = dict(topsoil=80.0, residuum=25.0, epikarst=300.0, rock=1500.0, void=1e4,
             topsoil_base=0.8, rock_depth=4.0, epikarst_thickness=2.0,
             sinkhole_x=35.0, sinkhole_depth=9.0, sinkhole_width=3.5,
             void_x=68.0, void_depth=9.0, void_width=5.0, void_height=3.0)


def ellipse(X, Z, x0, z0, width, height):
    return ((X - x0) / (width / 2)) ** 2 + ((Z - z0) / (height / 2)) ** 2 < 1


def cave_model(X, Z, soil, clay, rock, void, soil_base, clay_base,
               cave_x, cave_depth, cave_width, cave_height):
    rho = np.where(Z < soil_base, soil, np.where(Z < clay_base, clay, rock))
    return np.where(ellipse(X, Z, cave_x, cave_depth, cave_width, cave_height), void, rho)


def top_of_rock(x, rock_depth, sinkhole_x, sinkhole_depth, sinkhole_width):
    """Depth to rock: pinnacles and cutters, plus a sinkhole throat."""
    return (rock_depth + 1.2 * np.sin(2 * np.pi * x / 17) + 0.8 * np.sin(2 * np.pi * x / 7 + 1)
            + sinkhole_depth * np.exp(-((x - sinkhole_x) / sinkhole_width) ** 2))


def karst_model(X, Z, topsoil, residuum, epikarst, rock, void, topsoil_base, rock_depth,
                epikarst_thickness, sinkhole_x, sinkhole_depth, sinkhole_width,
                void_x, void_depth, void_width, void_height):
    top = top_of_rock(X, rock_depth, sinkhole_x, sinkhole_depth, sinkhole_width)
    rho = np.where(Z < top, residuum, np.where(Z < top + epikarst_thickness, epikarst, rock))
    rho = np.where(Z < topsoil_base, topsoil, rho)
    return np.where(ellipse(X, Z, void_x, void_depth, void_width, void_height), void, rho)


MODELS = {
    "cave": dict(rho=lambda X, Z: cave_model(X, Z, **CAVE),
                 title="cave in layered ground", clim=(30, 3000), profile_x=CAVE["cave_x"]),
    "karst": dict(rho=lambda X, Z: karst_model(X, Z, **KARST),
                  title="soil over karsted limestone", clim=(15, 3000),
                  profile_x=KARST["sinkhole_x"]),
}


def plot_true(ax, name, colorbar=True):
    """The true model, drawn finely, over the area of the model grid."""
    m = MODELS[name]
    xs = np.linspace(inv_mesh.x_edges[0], inv_mesh.x_edges[-1], 800)
    zs = np.linspace(0, inv_mesh.z_edges[-1], 250)
    X, Z = np.meshgrid(xs, zs)
    im = ax.pcolormesh(xs, zs, m["rho"](X, Z), cmap="turbo", norm=LogNorm(*m["clim"]),
                       shading="auto", rasterized=True)
    ex = survey.electrodes[:, 0]
    ax.plot(ex, 0 * ex, "kv", ms=3, clip_on=False)
    ax.set_xlim(xs[0], xs[-1]); ax.set_ylim(zs[-1], 0); ax.set_aspect("equal")
    ax.set_xlabel("position along line (m)"); ax.set_ylabel("depth (m)")
    if colorbar:
        ax.figure.colorbar(im, ax=ax, label="resistivity (Ω·m)", shrink=0.8, pad=0.02)
    return ax


fig, axes = plt.subplots(2, 1, figsize=(11, 6.5))
for ax, name in zip(axes, MODELS):
    plot_true(ax, name).set_title(f"true model: {MODELS[name]['title']}", fontsize=10)
plt.tight_layout(); plt.show()

## 3. The forward problem: synthetic data

The **forward problem** is: given the resistivity everywhere, what would
every reading be? `apparent_resistivity` solves it with a 2.5-D finite-element
model (the ground is 2-D, the current source is a point) and returns a copy of
the survey with `rhoa` filled in.

Two things make the test honest:

- **A different mesh.** The data are computed on a mesh four times finer than
  the model grid, with the true model sampled finely, so the cave's curved
  edges are not lined up with the cells the inversion solves for. Making and
  inverting data with exactly the same mesh is called the **inverse crime**:
  the inversion then looks better than it ever will on real data.
- **Noise.** Each reading is multiplied by $(1 + \varepsilon)$, with
  $\varepsilon$ drawn from a normal distribution with standard deviation
  `NOISE`. Real errors also come from electrode contact, electrode
  positions and 3-D ground, which are not random; this is the friendly case.

In [ ]:
fine_mesh = make_mesh(survey, dx=SPACING / 4, dz0=SPACING / 8, z_factor=1.06, depth=25)
rng = np.random.default_rng(SEED)

synth = {}
for name, m in MODELS.items():
    t0 = time.perf_counter()
    clean = apparent_resistivity(survey, m["rho"], mesh=fine_mesh)
    noisy = clean.copy()
    noisy.data["rhoa"] = clean.data["rhoa"] * (1 + NOISE * rng.standard_normal(len(clean)))
    noisy.data["r"] = noisy.data["rhoa"] / noisy.data["k"]
    synth[name] = {"clean": clean, "noisy": noisy}
    print(f"{name}: {len(clean)} readings in {time.perf_counter() - t0:.1f} s, "
          f"ρa from {clean.data['rhoa'].min():.0f} to {clean.data['rhoa'].max():.0f} Ω·m")

fig, axes = plt.subplots(3, 2, figsize=(15, 7.5), sharex=True)
for j, name in enumerate(MODELS):
    clean, noisy = synth[name]["clean"], synth[name]["noisy"]
    lo, hi = noisy.data["rhoa"].min(), noisy.data["rhoa"].max()
    clean.plot_pseudosection(axes[0, j], vmin=lo, vmax=hi, size=12)
    axes[0, j].set_title(f"{MODELS[name]['title']}: noise-free ρa", fontsize=10)
    noisy.plot_pseudosection(axes[1, j], vmin=lo, vmax=hi, size=12)
    axes[1, j].set_title(f"with {100 * NOISE:g} % noise: the 'observed' data", fontsize=10)
    added = 100 * (noisy.data["rhoa"] / clean.data["rhoa"] - 1)
    noisy.plot_pseudosection(axes[2, j], values=added, log=False, cmap="RdBu_r",
                             label="noise (%)", size=12)
    axes[2, j].set_title("the noise that was added", fontsize=10)
plt.tight_layout(); plt.show()

**Q2.** Where does the cave show up in the cave pseudosection: at what
positions and pseudo-depths? Is that where the cave is? Why does one body
make a pattern spread over so much of the pseudosection?

**Q3.** In the karst pseudosection, can you see the sinkhole throat at 35 m?
The small void at 68 m? What does that suggest the inversion will be able to
find?

## 4. The inversion and the smoothness parameter λ

The inversion looks for the resistivity $\rho_j$ of every cell of the model
grid. It works with $m_j = \ln \rho_j$ (so resistivities stay positive, and a
factor of two counts the same at 10 and at 1000 Ω·m) and minimises

$$
\Phi(\mathbf m) \;=\;
\underbrace{\sum_i \left(\frac{\ln \rho_{a,i}^{\text{obs}} - \ln \rho_{a,i}(\mathbf m)}{e_i}\right)^2}_{\Phi_d:\ \text{data misfit}}
\;+\; \lambda \,
\underbrace{\lVert \mathbf R\,\mathbf m \rVert^2}_{\text{roughness}}
$$

- $e_i$ is the **relative error** we assume for each reading (here the true
  noise level, 3 %).
- $\mathbf R$ takes the difference between every pair of neighbouring cells,
  so $\lVert \mathbf R \mathbf m \rVert^2$ is large for a blocky, rough
  image. Vertical differences are weighted by `Z_WEIGHT` = 0.5, which lets
  resistivity change faster with depth than along the line.
- **λ** sets the trade-off. Small λ: fit the data as closely as possible,
  whatever the image looks like. Large λ: keep the image smooth, even if
  that means fitting the data badly.

The misfit is reported as $\chi^2 = \Phi_d / N$ for $N$ readings. If the
predicted data are off by about the noise and no more, each term is about 1
and so is $\chi^2$.

- $\chi^2 \gg 1$: the model misses something the data say (**underfitting**).
- $\chi^2 \ll 1$: the model is fitting the noise (**overfitting**), and the
  extra detail it grows to do so is not real.

The problem is non-linear, so each inversion takes several **Gauss-Newton**
steps, each one linearising the forward problem around the current model.
Usually you stop as soon as $\chi^2$ reaches 1. Here every inversion runs to
convergence (`target_chi2=0`) so that λ alone decides the result.

The next cell inverts each model's noisy data for nine values of λ, from
0.1 to 1000. **It takes about three minutes**; the slider in section 5 then
just looks up the results.

In [ ]:
LAMS = [0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000]
Z_WEIGHT = 0.5
REL_ERROR = max(NOISE, 0.005)   # the error we assume: here, the noise we added
MAX_ITER = 10
SETTINGS = dict(mesh=inv_mesh, z_weight=Z_WEIGHT, rel_error=REL_ERROR,
                max_iter=MAX_ITER, target_chi2=0)


def true_on_grid(rho, mesh, sub=4):
    """The true model averaged (in log) over each model-grid cell."""
    f = (np.arange(sub) + 0.5) / sub
    xs = (mesh.x_edges[:-1, None] + np.diff(mesh.x_edges)[:, None] * f).ravel()
    zs = (mesh.z_edges[:-1, None] + np.diff(mesh.z_edges)[:, None] * f).ravel()
    X, Z = np.meshgrid(xs, zs)
    nz, nx = mesh.shape
    return 10 ** np.log10(rho(X, Z)).reshape(nz, sub, nx, sub).mean(axis=(1, 3))


TRUE_ON_GRID = {name: true_on_grid(m["rho"], inv_mesh) for name, m in MODELS.items()}


def model_error(res, name):
    """How far the image is from the truth: the average factor between them,
    over the cells the data cover (a perfect image scores 1)."""
    keep = res.covered()
    return float(10 ** np.abs(np.log10(res.resistivity / TRUE_ON_GRID[name]))[keep].mean())


sweeps = {}
for name in MODELS:
    t0 = time.perf_counter()
    sw = lambda_sweep(synth[name]["noisy"], lams=LAMS, **SETTINGS)
    sw["model_error"] = [model_error(r, name) for r in sw["result"]]
    sweeps[name] = sw
    print(f"{MODELS[name]['title']}: {len(LAMS)} inversions in "
          f"{time.perf_counter() - t0:.0f} s")
    display(sw[["lam", "chi2", "rms_pct", "roughness", "model_error"]].round(2))

In each table, χ² rises with λ (smoother images fit worse) and the roughness
falls. The last column, **model error**, is something you can only compute in
a synthetic test: on average, by what factor is the image wrong?

## 5. Explore: the smoothness slider

Move the **λ** slider and choose a **model**. The panels:

| left column | right column |
|---|---|
| the **true** model | a vertical **profile** through both at the dashed line (move it with the *profile x* slider) |
| the **inverted** model, faded where the data barely reach | χ² and model error against λ, the current λ starred |
| the **observed** (noisy) apparent resistivities | observed against predicted ρa, with the ±2 × noise band dotted |
| the **predicted** apparent resistivities, from the inverted model | the misfit of each reading divided by its error, against the bell curve it should follow if the model fits to the noise |
| the **misfit** of each reading divided by its error | |

If no sliders appear (for instance when viewing the notebook on GitHub),
call `explore("cave", 30, 47, True)` in a new cell instead.

In [ ]:
def plain_log_ticks(*axes, subs=(1.0,)):
    """Label log axes 10, 100, 1000 rather than 10¹, 10², 10³, and drop minor labels."""
    for ax in axes:
        for axis in (ax.xaxis, ax.yaxis):
            if axis.get_scale() == "log":
                axis.set_major_locator(LogLocator(base=10, subs=subs))
            axis.set_major_formatter(FormatStrFormatter("%g"))
            axis.set_minor_formatter(NullFormatter())


def plot_comparison(name, res, profile_x=None, fade=True, sweep=None):
    """True vs inverted model and observed vs predicted data for one inversion."""
    m = MODELS[name]
    lam = res.settings["lam"]
    dat = res.data.data
    obs, pred, err = (dat[c].to_numpy() for c in ("rhoa", "predicted", "error"))
    norm_misfit = np.log(obs / pred) / err
    profile_x = m["profile_x"] if profile_x is None else profile_x

    fig = plt.figure(figsize=(15, 12))
    gs = fig.add_gridspec(5, 2, width_ratios=[2.6, 1], hspace=0.6, wspace=0.25)
    ax_true, ax_inv, ax_obs, ax_pred, ax_mis = (fig.add_subplot(gs[i, 0]) for i in range(5))
    ax_prof = fig.add_subplot(gs[0:2, 1])
    ax_lam, ax_x, ax_h = (fig.add_subplot(gs[i, 1]) for i in (2, 3, 4))

    # models
    plot_true(ax_true, name).set_title(f"true model: {m['title']}", fontsize=10)
    res.plot(ax_inv, vmin=m["clim"][0], vmax=m["clim"][1], mask=fade)
    ax_inv.set_title(f"inverted, λ = {lam:g}: χ² = {res.chi2:.2f}, "
                     f"off by ×{model_error(res, name):.2f} on average", fontsize=10)
    for ax in (ax_true, ax_inv):
        ax.axvline(profile_x, color="w", lw=1.2, ls="--")

    # data
    lo, hi = obs.min(), obs.max()
    res.data.plot_pseudosection(ax_obs, vmin=lo, vmax=hi, size=12)
    ax_obs.set_title("observed apparent resistivity (synthetic, with noise)", fontsize=10)
    res.data.plot_pseudosection(ax_pred, values=pred, vmin=lo, vmax=hi, size=12,
                                label="apparent resistivity (Ω·m)")
    ax_pred.set_title("predicted by the inverted model", fontsize=10)
    res.data.plot_pseudosection(ax_mis, values=norm_misfit, log=False, cmap="RdBu_r",
                                vmin=-3, vmax=3, size=12, label="misfit / error")
    ax_mis.set_title(f"misfit ÷ error (RMS misfit {res.rms:.1f} %)", fontsize=10)

    # profile
    zf = np.linspace(0, inv_mesh.z_edges[-1], 500)
    ax_prof.semilogx(m["rho"](np.full_like(zf, profile_x), zf), zf, "k-", lw=1.5,
                     label="true")
    p = res.profile(profile_x)
    seen = p["coverage"] >= 0.01
    ax_prof.semilogx(p["resistivity"], p["depth"], "-", color="C3", lw=0.8, alpha=0.4)
    ax_prof.semilogx(p["resistivity"][seen], p["depth"][seen], "o-", color="C3", ms=3,
                     label="inverted")
    ax_prof.set_xlim(m["clim"][0] / 2, max(m["clim"][1], 2e4))
    ax_prof.set_ylim(zf[-1], 0)
    ax_prof.set_xlabel("resistivity (Ω·m)"); ax_prof.set_ylabel("depth (m)")
    ax_prof.set_title(f"profile at x = {profile_x:.0f} m", fontsize=10)
    ax_prof.legend(fontsize=8, loc="lower left")

    # trade-off
    if sweep is not None:
        ax_lam.loglog(sweep["lam"], sweep["chi2"], "o-", color="C0", ms=4)
        ax_e = ax_lam.twinx()
        ax_e.semilogx(sweep["lam"], sweep["model_error"], "s-", color="C1", ms=4)
        ax_e.plot(lam, model_error(res, name), "*", color="C1", ms=14, mec="k")
        ax_e.set_ylabel("model error (×)", color="C1")
    ax_lam.plot(lam, res.chi2, "*", color="C0", ms=14, mec="k")
    ax_lam.axhline(1, color="k", ls="--", lw=0.8)
    ax_lam.set_xlabel("λ"); ax_lam.set_ylabel("χ²", color="C0")
    ax_lam.set_title("data misfit and model error", fontsize=10)

    # observed vs predicted
    ax_x.loglog(obs, pred, ".", ms=3, color="C0")
    lim = np.array([lo * 0.8, hi * 1.25])
    ax_x.plot(lim, lim, "k-", lw=0.8)
    for s in (-2, 2):
        ax_x.plot(lim, lim * (1 + s * NOISE), "k:", lw=0.8)
    ax_x.set_xlim(lim); ax_x.set_ylim(lim)
    ax_x.set_xlabel("observed ρa (Ω·m)"); ax_x.set_ylabel("predicted ρa (Ω·m)")
    plain_log_ticks(ax_lam)
    plain_log_ticks(ax_x, subs=(1.0, 2.0, 5.0))

    # misfit histogram
    bins = np.linspace(-5, 5, 41)
    ax_h.hist(np.clip(norm_misfit, -5, 5), bins=bins, density=True, color="C0", alpha=0.7)
    g = np.linspace(-5, 5, 200)
    ax_h.plot(g, np.exp(-g ** 2 / 2) / np.sqrt(2 * np.pi), "k-", lw=1)
    ax_h.set_xlabel("misfit ÷ error")
    ax_h.set_title(f"spread {norm_misfit.std():.2f} (1 = fits to the noise)", fontsize=10)
    plt.show()


def explore(model, lam, profile_x, fade):
    sw = sweeps[model]
    res = sw.loc[sw["lam"] == lam, "result"].iloc[0]
    plot_comparison(model, res, profile_x=profile_x, fade=fade, sweep=sw)


style = {"description_width": "initial"}
w_model = Dropdown(options=[(m["title"], k) for k, m in MODELS.items()], value="cave",
                   description="model", style=style)
w_lam = SelectionSlider(options=LAMS, value=LAMS[len(LAMS) // 2],
                        description="λ (smoothness)", continuous_update=False, style=style)
w_x = FloatSlider(value=MODELS["cave"]["profile_x"], min=float(survey.electrodes[0, 0]),
                  max=float(survey.electrodes[-1, 0]), step=1, description="profile x (m)",
                  continuous_update=False, style=style)
w_fade = Checkbox(value=True, description="fade cells the data barely see")
w_model.observe(lambda ch: setattr(w_x, "value", MODELS[ch["new"]]["profile_x"]), names="value")

interact(explore, model=w_model, lam=w_lam, profile_x=w_x, fade=w_fade);

**Q4.** Start at the smallest λ and step up to the largest, for the cave.
Describe what happens to (a) the inverted image, (b) χ², (c) the predicted
pseudosection, and (d) the misfit pseudosection. At small λ, is the misfit
pattern random speckle or does it have structure? At large λ?

**Q5.** At the smallest λ the image fits the data *better* than the noise
level (χ² < 1, the histogram narrower than the bell curve). Which features of
that image are not in the true model? Why are they there?

**Q6.** For the karst model, move the profile through the sinkhole (35 m),
through a pinnacle (around 46 m) and through the void (68 m). Which of these
does the inversion find at any λ? Which does it miss at every λ?

**Q7.** Untick *fade*. What does the inversion put where the data do not
reach (the bottom corners)? Where does that come from?

## 6. Choosing λ

In a synthetic test you can pick the λ with the smallest model error. With
field data you cannot, so you need a rule that uses only the data. Two are
common:

- **The discrepancy principle**: the smoothest model that fits the data to
  their noise level, χ² ≈ 1. It needs a good estimate of the noise (from
  repeat or reciprocal readings).
- **The L-curve**: plot misfit against roughness on log axes. It usually has
  a corner, where making the image rougher stops buying much fit. It needs no
  noise estimate, but the corner is not always clear.

The cell below draws both, finds where χ² crosses 1, and compares it with the
λ the truth would have chosen.

In [ ]:
def lambda_at_chi2_one(sw):
    """λ where χ² = 1, interpolated in log-log (None if never crossed)."""
    lam = np.log10(sw["lam"].to_numpy(float))
    chi = np.maximum.accumulate(np.log10(sw["chi2"].to_numpy(float)))
    if chi[0] > 0 or chi[-1] < 0:
        return None
    return float(10 ** np.interp(0, chi, lam))


best_lam = {}
for name, sw in sweeps.items():
    lam1 = lambda_at_chi2_one(sw)
    best = sw.loc[sw["model_error"].idxmin()]
    pick = sw.iloc[np.argmin(np.abs(np.log(sw["lam"] / (lam1 or best["lam"]))))]
    best_lam[name] = pick["lam"]
    lam1_text = "never" if lam1 is None else f"at λ ≈ {lam1:.0f}"
    print(f"{MODELS[name]['title']}: χ² = 1 {lam1_text}; smallest model error at "
          f"λ = {best['lam']:g} (×{best['model_error']:.2f})")

    fig = plt.figure(figsize=(12, 11))
    gs = fig.add_gridspec(4, 2, height_ratios=[1.3, 1, 1, 1], hspace=0.5)
    a0, a1 = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
    a0.loglog(sw["roughness"], sw["chi2"], "o-")
    for _, r in sw.iterrows():
        a0.annotate(f" λ={r['lam']:g}", (r["roughness"], r["chi2"]), fontsize=8)
    a0.axhline(1, color="k", ls="--", lw=0.8)
    a0.set_xlabel("roughness ‖Rm‖"); a0.set_ylabel("χ²")
    a0.set_title("L-curve", fontsize=10)
    a1.loglog(sw["lam"], sw["chi2"], "o-", label="χ² (data)")
    a1.loglog(sw["lam"], sw["model_error"], "s-", label="model error (truth)")
    a1.axhline(1, color="k", ls="--", lw=0.8)
    if lam1:
        a1.axvline(lam1, color="C0", ls=":", lw=1)
    a1.set_xlabel("λ"); a1.legend(fontsize=8)
    a1.set_title("misfit and model error against λ", fontsize=10)
    plain_log_ticks(a0, a1)
    for k, lam in enumerate([sw["lam"].iloc[0], pick["lam"], sw["lam"].iloc[-1]]):
        r = sw.loc[sw["lam"] == lam].iloc[0]
        ax = fig.add_subplot(gs[k + 1, :])
        r["result"].plot(ax, vmin=MODELS[name]["clim"][0], vmax=MODELS[name]["clim"][1])
        ax.set_title(f"λ = {lam:g}: χ² = {r['chi2']:.2f}, model error ×{r['model_error']:.2f}",
                     fontsize=10)
    fig.suptitle(MODELS[name]["title"])
    plt.show()

**Q8.** How close is the λ from the discrepancy principle (χ² = 1) to the λ
with the smallest model error? Is the corner of the L-curve near either? The
model error grows on *both* sides of its minimum: compare how fast it grows
when λ is too small with how fast it grows when λ is too large.

**Q9.** The discrepancy principle usually lands on the smooth side of the best
λ. Is that the safe side to be on? Think about what each kind of error makes
you interpret: features that are not there, or features you miss.

**Q10.** The discrepancy principle needs the noise level. What would happen
to the λ it picks if you assumed 1 % error when the real noise is 3 %? If you
assumed 10 %? (Section 8 lets you test it.)

## 7. What a smooth inversion cannot recover

Even at the best λ, the image is not the ground. Three limits show up in
every ERT survey:

1. **Contrast saturates.** Once a body is ten or so times more resistive
   than its surroundings, current already goes around it, and making it more
   resistive barely changes the data. The data cannot tell 5 000 Ω·m from
   50 000 Ω·m, so the inversion reports the smoothest value consistent with
   them, which is far less than the truth.
2. **Smoothing spreads features.** A sharp-edged body comes back as a blur,
   larger and weaker than it is; sharp layer boundaries come back as
   gradients.
3. **Sensitivity falls with depth and towards the ends of the line.** A body
   must be larger the deeper it is to be seen; at the depth of the karst void
   (9 m), a 5 m body is close to the limit.

In [ ]:
rows = []
for name, sw in sweeps.items():
    res = sw.loc[sw["lam"] == best_lam[name], "result"].iloc[0]
    X, Z = inv_mesh.mesh()
    if name == "cave":
        inside = ellipse(X, Z, CAVE["cave_x"], CAVE["cave_depth"],
                         CAVE["cave_width"], CAVE["cave_height"])
        true_void, host = CAVE["void"], CAVE["rock"]
    else:
        inside = ellipse(X, Z, KARST["void_x"], KARST["void_depth"],
                         KARST["void_width"], KARST["void_height"])
        true_void, host = KARST["void"], KARST["rock"]
    rows.append({"model": MODELS[name]["title"], "λ": best_lam[name],
                 "void: true (Ω·m)": true_void, "host rock (Ω·m)": host,
                 "inverted, highest in void": res.resistivity[inside].max(),
                 "inverted, mean in void": np.exp(np.log(res.resistivity[inside]).mean())})
display(pd.DataFrame(rows).round(0))

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5))
for ax, name in zip(axes, MODELS):
    res = sweeps[name].loc[sweeps[name]["lam"] == best_lam[name], "result"].iloc[0]
    res.plot(ax, field="coverage")
    ax.set_title(f"{MODELS[name]['title']}: how much the data see each cell "
                 "(log scale)", fontsize=10)
plt.tight_layout(); plt.show()

**Q11.** By what factor does the inversion underestimate the cave's
resistivity? Would a geologist reading the image alone conclude "air-filled
void" or "slightly more resistive limestone"? What other information would
help decide?

**Q12.** Compare the coverage plot with the karst void's position. Why is the
void hard to find even though it is as resistive as the cave? The inverted
resistivity where the void is comes out *lower* than the true limestone
around it: where does that low value come from?

## 8. Your turn

### A live inversion

Pick any λ and vertical smoothing weight and press **Run inversion**. Each run
takes about ten seconds. The trade-off panel still shows the section 4 sweep
(run with `Z_WEIGHT` = 0.5) for reference, with your run starred.

`z_weight` below 1 favours layers (resistivity may change quickly with
depth); above 1 favours vertical structure such as pinnacles and cutters.

In [ ]:
def live(model, lam, z_weight):
    t0 = time.perf_counter()
    res = resistivity_tomography(synth[model]["noisy"], lam=lam,
                                 **{**SETTINGS, "z_weight": z_weight})
    print(f"λ = {lam:.3g}, z_weight = {z_weight:.2g}: {len(res.history) - 1} iterations, "
          f"{time.perf_counter() - t0:.1f} s")
    plot_comparison(model, res, sweep=sweeps[model])


interact_manual.options(manual_name="Run inversion")(
    live,
    model=Dropdown(options=[(m["title"], k) for k, m in MODELS.items()], value="karst",
                   description="model", style=style),
    lam=FloatLogSlider(value=30, base=10, min=-1, max=4, step=0.1, description="λ",
                       style=style),
    z_weight=FloatLogSlider(value=Z_WEIGHT, base=10, min=-1, max=1, step=0.1,
                            description="z_weight", style=style),
);

### Things to try

Most of these mean changing a setting near the top and `Runtime > Run all`
(about four minutes).

1. **Noise level.** Set `NOISE` to 0.01, then 0.10. How does the λ where
   χ² = 1 move? Which features of each model survive the noisier data?
2. **Assuming the wrong error.** Keep `NOISE = 0.03` but set
   `REL_ERROR = 0.01` (in section 4), then 0.10. Where does χ² = 1 fall now,
   and what does the discrepancy principle's image look like? This is the
   situation with field data whenever the noise estimate is wrong.
3. **Array.** Set `ARRAY = "wenner"`, then `"schlumberger"`. Which array
   finds the cave best? The layering? The sinkhole? Compare the number of
   readings and the pseudo-depths.
4. **Vertical smoothing.** In the live inversion, use the karst model with
   `z_weight` 0.2, 1 and 3 at the same λ. Which images the pinnacles best?
   Does χ² tell you which is right?
5. **How deep can the cave be?** In `CAVE`, set `cave_depth` to 8, 10 and 12
   (keep the size). At what depth does it disappear? Is that deeper or
   shallower than the line length divided by five, the usual rule of thumb?
6. **How resistive?** Set the cave's `void` to 2 000, then 100 000. How much
   do the data change (look at the pseudosections)? How much does the image?
7. **A water- or clay-filled cave.** Set the cave's `void` to 20 Ω·m. Is a
   conductive cave easier or harder to find than an air-filled one of the
   same size? Why might that be?
8. **Your own model.** Write a new `rho(X, Z)` function — a buried pipe, a
   dipping contact, a landfill — add it to `MODELS` with a title, colour
   limits and a profile position, and run everything.

**Q13.** Pick the experiment whose result surprised you most and explain it.

## Summary

- A **synthetic test** — known model, forward-modelled data, realistic noise,
  inversion, comparison — is the only way to find out what a survey can and
  cannot resolve. Run one with the same electrodes and array before
  interpreting field data.
- The data do not decide one image: **λ chooses** among the many that fit.
  Too small and the image grows artefacts that fit the noise (χ² < 1); too
  large and it misses what the data say (χ² ≫ 1, structured misfit).
- **χ² ≈ 1**, with a good noise estimate, picks a λ near the best one, on the
  smooth side; the misfit should then look like random speckle, not
  patterns.
- Even the best image **underestimates strong contrasts, blurs edges** and
  fades with depth: an air-filled void looks like slightly resistive rock,
  and small deep bodies disappear.